# Tome C2, Chapter 3: Data Structures (Python companion)

Exercises first, solutions at the end. We build each structure of the chapter from scratch and measure it against its theorem: heap construction against $2n$, AVL heights against Fibonacci, collision rates of universal hashing against $1/m$ (and a fixed hash function against an adversary), FKS perfect hashing, union–find with counters against $\log^*n$, and skip-list searches against $2\log_2 n$.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, heapq
import numpy as np
import matplotlib.pyplot as plt
random.seed(3)
lg = math.log2

## Exercises

**P3.1 (★) Heaps.** Implement a binary heap with comparison counters. Check Proposition 3.2 (Floyd's construction uses fewer than $2n$ comparisons) and compare with building by $n$ insertions; sort with heapsort and compare its comparison count with merge sort's $W(n)$.

**P3.2 (★★) AVL trees.** Implement AVL insertion with the rotations of Theorem 3.5. Insert keys in sorted order and in random order and compare the heights with $\log_2 n$ and with the bound $\log_\varphi(n+1)-1$ of Theorem 3.4. Build the Fibonacci trees and check that they are AVL trees of minimum size.

**P3.3 (★) Hashing.** Measure collision rates of the Carter–Wegman family on adversarial keys and compare with $1/m$ and with $x\bmod m$. Draw the distribution of chain lengths. Check the counterexample of Exercise 3.4.

**P3.4 (★★) Perfect hashing.** Implement the two-level scheme of Theorem 3.9 and measure its space $\sum b_i^2$ and the number of retries.

**P3.5 (★★) Union–find.** Implement union by rank and path compression with a step counter. Run random sequences of operations and adversarial ones, compare the steps per operation with $\log^* n$, and check Exercises 3.7 and 3.8.

**P3.6 (★) Skip lists.** Implement a skip list, measure the average search length and the number of nodes, and compare with Theorem 3.12.

**P3.7 (★) Running median.** Solve Exercise 3.10 with two heaps and check it against sorting.

## Solutions

### P3.1 Heaps

In [ ]:
class Heap:
    def __init__(self, data=None):
        self.a, self.c = [], 0
        if data is not None:
            self.a = list(data)
            for i in range(len(self.a) // 2 - 1, -1, -1): self._down(i)
    def _less(self, i, j): self.c += 1; return self.a[i] < self.a[j]
    def _up(self, i):
        while i > 0 and self._less(i, (i - 1) // 2):
            p = (i - 1) // 2; self.a[i], self.a[p] = self.a[p], self.a[i]; i = p
    def _down(self, i):
        n = len(self.a)
        while 2 * i + 1 < n:
            c = 2 * i + 1
            if c + 1 < n and self._less(c + 1, c): c += 1
            if not self._less(c, i): break
            self.a[i], self.a[c] = self.a[c], self.a[i]; i = c
    def push(self, x): self.a.append(x); self._up(len(self.a) - 1)
    def pop(self):
        top, last = self.a[0], self.a.pop()
        if self.a: self.a[0] = last; self._down(0)
        return top

W = lambda n: sum(math.ceil(lg(k)) for k in range(1, n + 1))
for n in (1000, 10000, 100000):
    xs = [random.random() for _ in range(n)]
    h = Heap(xs); floyd = h.c
    g = Heap()
    for x in xs: g.push(x)
    ins = g.c
    h.c = 0; out = [h.pop() for _ in range(n)]; assert out == sorted(xs)
    print(f"n={n:6d}: Floyd {floyd/n:.2f}n comparisons, by insertions {ins/n:.2f}n;  heapsort extractions {h.c/(n*lg(n)):.2f} n log2 n "
          f"(merge sort worst case {W(n)/(n*lg(n)):.2f} n log2 n)")

Floyd's construction uses about $1.88n$ comparisons on random input, below the bound $2n$; random insertions are not much worse (a random new key rarely climbs far), but inserting keys in decreasing order would cost about $n\log_2 n$. Heapsort's extractions cost about $1.7n\log_2n$ here, slowly approaching $2n\log_2 n$, twice merge sort: the price of sifting a large leaf down from the root, two comparisons per level.

### P3.2 AVL trees

In [ ]:
class Node:
    __slots__ = ("key", "left", "right", "h")
    def __init__(self, key): self.key, self.left, self.right, self.h = key, None, None, 0
H = lambda t: t.h if t else -1
def fix(t): t.h = 1 + max(H(t.left), H(t.right)); return t
def rot_right(z):
    y = z.left; z.left = y.right; y.right = fix(z); return fix(y)
def rot_left(z):
    y = z.right; z.right = y.left; y.left = fix(z); return fix(y)
def insert(t, key, stats):
    if t is None: return Node(key)
    if key < t.key: t.left = insert(t.left, key, stats)
    else: t.right = insert(t.right, key, stats)
    fix(t)
    if H(t.left) - H(t.right) == 2:
        if key < t.left.key: stats["single"] += 1; return rot_right(t)
        stats["double"] += 1; t.left = rot_left(t.left); return rot_right(t)
    if H(t.right) - H(t.left) == 2:
        if key >= t.right.key: stats["single"] += 1; return rot_left(t)
        stats["double"] += 1; t.right = rot_right(t.right); return rot_left(t)
    return t
def check(t):
    if t is None: return -1
    a, b = check(t.left), check(t.right)
    assert abs(a - b) <= 1 and t.h == 1 + max(a, b)
    return t.h

import sys; sys.setrecursionlimit(10000)
phi = (1 + 5 ** 0.5) / 2
for order in ("sorted", "random"):
    for n in (1000, 100000):
        keys = list(range(n)) if order == "sorted" else random.sample(range(10 ** 9), n)
        t, st = None, {"single": 0, "double": 0}
        for k in keys: t = insert(t, k, st)
        check(t)
        print(f"{order:6s} n={n:6d}: height {t.h:2d}, log2 n = {lg(n):5.2f}, bound {math.log(n+1, phi)-1:5.2f}; rotations {st}")

Sorted insertions, the worst case for a plain search tree, give an AVL tree of height $\lfloor\log_2 n\rfloor$: perfectly balanced. Random insertions give height about $1.04\log_2 n$. The bound $1.44\log_2 n$ is reached only by the Fibonacci trees.

In [ ]:
def fib_tree(h, start=[0]):
    if h < 0: return None
    if h == 0:
        t = Node(start[0]); start[0] += 1; return t
    left = fib_tree(h - 1, start); t = Node(start[0]); start[0] += 1; right = fib_tree(h - 2, start)
    t.left, t.right = left, right; return fix(t)
size = lambda t: 0 if t is None else 1 + size(t.left) + size(t.right)
F = [0, 1]
while len(F) < 40: F.append(F[-1] + F[-2])
for h in range(0, 20, 3):
    t = fib_tree(h, [0]); check(t)
    assert size(t) == F[h + 3] - 1
    print(f"height {h:2d}: Fibonacci tree with {size(t):5d} = F_{h+3}-1 nodes; log_phi(n+1)-1 = {math.log(size(t)+1, phi)-1:.2f}")

### P3.3 Hashing

An adversary who knows that we use $x\bmod m$ sends multiples of $m$. Against the Carter–Wegman family, the same keys collide at rate at most $1/m$, whatever they are.

In [ ]:
p = (1 << 61) - 1          # a Mersenne prime
m = 1024
def cw():
    a, b = random.randrange(1, p), random.randrange(p)
    return lambda x: ((a * x + b) % p) % m
keys = [m * i for i in range(2000)]                    # adversarial for x mod m
pairs = [(random.choice(keys), random.choice(keys)) for _ in range(20000)]
pairs = [(x, y) for x, y in pairs if x != y]
coll_mod = sum((x % m) == (y % m) for x, y in pairs) / len(pairs)
coll_cw = 0
for x, y in pairs:
    h = cw(); coll_cw += h(x) == h(y)
print(f"collision rate: x mod m {coll_mod:.3f};  Carter-Wegman {coll_cw/len(pairs):.5f};  1/m = {1/m:.5f}")

h = cw(); n = m
for name, ks in (("random keys", random.sample(range(10 ** 15), n)), ("multiples of m", keys[:n])):
    loads = np.bincount([h(x) for x in ks], minlength=m)
    dist = {int(k): int(v) for k, v in zip(*np.unique(loads, return_counts=True))}
    print(f"{name:15s} chain lengths (length: buckets) {dist}; sum of squares / n = {(loads ** 2).sum() / n:.3f}")

coll = sum(((a * 1) % 5) % 3 == ((a * 4) % 5) % 3 for a in range(1, 5))
print("Exercise 3.4: without b, keys 1 and 4 collide for", coll, "of the 4 values of a")

For random keys the chain lengths follow a Poisson law of mean $1$: about $37\%$ of empty buckets, $37\%$ with one key, $18\%$ with two, and so on, and $\sum b_i^2/n$ is close to $2$, as in the proof of Theorem 3.9. The structured keys $0,m,2m,\dots$ are spread *more* evenly than random ones: $(ax+b)\bmod p$ maps an arithmetic progression to an arithmetic progression. Universality only promises the collision bound for each pair, not randomness.

### P3.4 Perfect hashing

In [ ]:
def fks(S):
    n = len(S); tries1 = 0
    while True:
        tries1 += 1; a, b = random.randrange(1, p), random.randrange(p)
        h = lambda x, a=a, b=b: ((a * x + b) % p) % n
        buckets = [[] for _ in range(n)]
        for x in S: buckets[h(x)].append(x)
        if sum(len(B) ** 2 for B in buckets) < 4 * n: break
    tables, tries2 = [], 0
    for B in buckets:
        mb = len(B) ** 2
        if mb == 0: tables.append((None, [])); continue
        while True:
            tries2 += 1; a2, b2 = random.randrange(1, p), random.randrange(p)
            cells = [None] * mb; ok = True
            for x in B:
                j = ((a2 * x + b2) % p) % mb
                if cells[j] is not None: ok = False; break
                cells[j] = x
            if ok: tables.append(((a2, b2, mb), cells)); break
    def lookup(x):
        par, cells = tables[h(x)]
        if par is None: return False
        a2, b2, mb = par
        return cells[((a2 * x + b2) % p) % mb] == x
    space = n + sum(len(c) for _, c in tables)
    nonempty = sum(1 for B in buckets if B)
    return lookup, space, tries1, tries2 / nonempty

for n in (1000, 10000, 100000):
    S = random.sample(range(10 ** 12), n)
    lookup, space, t1, t2 = fks(S)
    assert all(lookup(x) for x in S[:2000]) and not any(lookup(x) for x in random.sample(range(10 ** 12, 2 * 10 ** 12), 2000))
    print(f"n={n:6d}: space {space/n:.2f} n cells, level-one draws {t1}, level-two draws per non-empty bucket {t2:.2f}")

### P3.5 Union–find

`steps` counts parent pointers followed by finds (the quantity bounded in Theorem 3.11).

In [ ]:
class UF:
    def __init__(self, n, by_rank=True, compress=True):
        self.p, self.r, self.steps = list(range(n)), [0] * n, 0
        self.by_rank, self.compress = by_rank, compress
    def find(self, x):
        root = x
        while self.p[root] != root: root = self.p[root]; self.steps += 1
        if self.compress:
            while self.p[x] != root: self.p[x], x = root, self.p[x]
        return root
    def union(self, x, y):
        x, y = self.find(x), self.find(y)
        if x == y: return
        if self.by_rank:
            if self.r[x] > self.r[y]: x, y = y, x
            if self.r[x] == self.r[y]: self.r[y] += 1
        self.p[x] = y

def logstar(n):
    k = 0
    while n > 1: n = lg(n); k += 1
    return k

for n in (10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6):
    row = []
    for br, cp in ((True, True), (True, False), (False, True)):
        U = UF(n, br, cp); m = 0
        for _ in range(2 * n):
            x, y = random.randrange(n), random.randrange(n)
            if random.random() < 0.5: U.union(x, y)
            else: U.find(x)
            m += 1
        row.append(U.steps / m)
    print(f"n={n:8d}: steps/op  rank+compression {row[0]:.3f}   rank only {row[1]:.3f}   compression only {row[2]:.3f}    log* n = {logstar(n)}")

On random operations all three variants look fast, and the full structure takes a fraction of a step per operation. The theorems are about adversaries; here are the bad sequences of Exercises 3.7 and 3.8.

In [ ]:
k = 16; n = 1 << k
U = UF(n, True, False)
size = 1
while size < n:
    for i in range(0, n, 2 * size): U.union(i, i + size)
    size *= 2
worst = 0
for x in range(n):
    U.steps = 0; U.find(x); worst = max(worst, U.steps)
print(f"rank only, n = 2^{k}: the longest find takes {worst} steps = log2 n")
n = 10 ** 4
U = UF(n, False, True)
for i in range(n - 1): U.union(i, i + 1)
U.steps = 0; U.find(0); first = U.steps; U.steps = 0; U.find(0)
print(f"compression only, path of {n}: first find {first} steps, second find {U.steps} steps")

### P3.6 Skip lists

In [ ]:
class SkipList:
    def __init__(self):
        self.head = [None]            # head[i] = first node at level i; a node is [key, next pointers...]
        self.nodes = 0
    def insert(self, key):
        lvl = 1
        while random.random() < 0.5: lvl += 1
        while len(self.head) < lvl: self.head.append(None)
        node = [key] + [None] * lvl; self.nodes += lvl
        cur = None                     # None stands for the sentinel
        for i in range(len(self.head) - 1, -1, -1):
            nxt = self.head[i] if cur is None else cur[1 + i]
            while nxt is not None and nxt[0] < key:
                cur = nxt; nxt = cur[1 + i]
            if i < lvl:
                node[1 + i] = nxt
                if cur is None: self.head[i] = node
                else: cur[1 + i] = node
    def search(self, key):
        steps, cur = 0, None
        for i in range(len(self.head) - 1, -1, -1):
            nxt = self.head[i] if cur is None else cur[1 + i]
            while nxt is not None and nxt[0] <= key:
                cur = nxt; nxt = cur[1 + i]; steps += 1
            steps += 1                 # the step down (or the final stop)
        return cur is not None and cur[0] == key, steps

for n in (1000, 10000, 100000):
    S = SkipList(); keys = random.sample(range(10 ** 9), n)
    for k in keys: S.insert(k)
    res = [S.search(k) for k in random.sample(keys, min(2000, n))]
    assert all(f for f, _ in res)
    print(f"n={n:6d}: average search {np.mean([s for _, s in res]):5.1f} steps, 2 log2 n + 4 = {2*lg(n)+4:5.1f};  nodes {S.nodes/n:.2f} n;  levels {len(S.head)}")

### P3.7 Running median

In [ ]:
def running_medians(stream):
    lo, hi, out = [], [], []          # lo is a max-heap (stored negated), hi a min-heap
    for x in stream:
        if lo and x > -lo[0]: heapq.heappush(hi, x)
        else: heapq.heappush(lo, -x)
        if len(lo) > len(hi) + 1: heapq.heappush(hi, -heapq.heappop(lo))
        elif len(hi) > len(lo): heapq.heappush(lo, -heapq.heappop(hi))
        out.append(-lo[0])
    return out
xs = [random.randrange(1000) for _ in range(3000)]
meds = running_medians(xs)
assert all(meds[i] == sorted(xs[:i + 1])[i // 2] for i in range(0, 3000, 7))
print("running medians agree with sorting; last median:", meds[-1])